# Food Price CPI — Data Exploration

This notebook registers and inspects all data sources used in the food price
CPI forecasting experiment:

1. **Statistics Canada food CPI series** — 9 categories from table 18-10-0004-11
2. **FRED exogenous covariates** — 7 US/Canadian economic indicators

Sections:
1. Register StatCan food CPI series and validate availability
2. Register FRED covariate series and validate availability
3. Inspect date ranges and coverage gaps
4. Visualise food CPI history and seasonal patterns
5. Visualise FRED covariates and correlations with food CPI

## Prerequisites

```bash
uv run python scripts/fetch_cpi.py
uv run python scripts/fetch_fred.py   # requires FRED_API_KEY in .env
```

In [ ]:
import sys
from pathlib import Path

# Notebook lives at implementations/experiments/food_price_forecasting/ — 3 levels deep.
repo_root = Path().resolve().parents[2]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## 1. Register StatCan Food CPI Series

In [ ]:
from aieng.forecasting.data import DataService, SeriesMetadata
from aieng.forecasting.data.adapters import StatCanAdapter

CPI_TABLE_ID = "18-10-0004-11"
CACHE_DIR = repo_root / "data" / "statcan"

# The 9 CFPR food categories + the broader food series used as covariates.
# Series IDs match those in reference_specs/food_cpi/*.yaml and scripts/fetch_cpi.py.
FOOD_CPI_SERIES = [
    ("cpi_food_canada", "Food", "CPI Food, Canada (2002=100)"),
    ("cpi_bakery_cereal_canada", "Bakery and cereal products (excluding baby food)", "CPI Bakery and cereal products (excl. baby food), Canada (2002=100)"),
    ("cpi_dairy_eggs_canada", "Dairy products and eggs", "CPI Dairy products and eggs, Canada (2002=100)"),
    ("cpi_fish_seafood_canada", "Fish, seafood and other marine products", "CPI Fish, seafood and other marine products, Canada (2002=100)"),
    ("cpi_restaurants_canada", "Food purchased from restaurants", "CPI Food purchased from restaurants, Canada (2002=100)"),
    ("cpi_fruit_preparations_nuts_canada", "Fruit, fruit preparations and nuts", "CPI Fruit, fruit preparations and nuts, Canada (2002=100)"),
    ("cpi_meat_canada", "Meat", "CPI Meat, Canada (2002=100)"),
    ("cpi_other_food_nonalcoholic_canada", "Other food products and non-alcoholic beverages", "CPI Other food products and non-alcoholic beverages, Canada (2002=100)"),
    ("cpi_vegetables_preparations_canada", "Vegetables and vegetable preparations", "CPI Vegetables and vegetable preparations, Canada (2002=100)"),
]

svc = DataService()
CACHE_DIR.mkdir(parents=True, exist_ok=True)

failed = []
for series_id, product_group, description in FOOD_CPI_SERIES:
    adapter = StatCanAdapter(
        table_id=CPI_TABLE_ID,
        member_filter={"GEO": "Canada", "Products and product groups": product_group},
        cache_dir=CACHE_DIR,
    )
    metadata = SeriesMetadata(
        series_id=series_id,
        description=description,
        source="StatCan",
        units="Index 2002=100",
        frequency="MS",
        table_id=CPI_TABLE_ID,
    )
    try:
        svc.register(series_id, adapter, metadata)
    except Exception as e:
        failed.append((series_id, str(e)))
        print(f"  [WARN] {series_id}: {e}")

if failed:
    print(f"\n{len(failed)} series failed to register. Check StatCan member filter labels above.")
    print("Run scripts/fetch_cpi.py and inspect the printed series list to find correct labels.")
else:
    print(f"Registered {len(FOOD_CPI_SERIES)} food CPI series.")

In [ ]:
import pandas as pd
from datetime import datetime, timezone

summary = svc.summary()
summary["start"] = summary["start"].dt.strftime("%Y-%m")
summary["end"] = summary["end"].dt.strftime("%Y-%m")
summary

## 2. Register FRED Covariate Series

Requires `FRED_API_KEY` in the environment.  Set it in `.env` and reload the kernel if needed.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(repo_root / ".env")

from aieng.forecasting.data.adapters import FREDAdapter

FRED_SERIES = [
    ("fred_us_cpi_food_at_home", "CPIFABSL", "US CPI: Food at Home (1982-84=100)", "Index 1982-84=100"),
    ("fred_us_cpi_meats_poultry_fish_eggs", "CUSR0000SAF112", "US CPI: Meats, Poultry, Fish, Eggs (1982-84=100)", "Index 1982-84=100"),
    ("fred_us_cpi_fruits_vegetables", "CUSR0000SAF113", "US CPI: Fruits and Vegetables (1982-84=100)", "Index 1982-84=100"),
    ("fred_canada_10yr_bond_yield", "IRLTLT01CAM156N", "Canada 10-year Government Bond Yield (% p.a.)", "Percent per annum"),
    ("fred_canada_us_exchange_rate", "EXCAUS", "Canada/US Exchange Rate (CAD per USD)", "CAD per USD"),
    ("fred_sp100_volatility_vxo", "VXOCLS", "S&P 100 Volatility Index (VXO)", "Index"),
    ("fred_wilshire_5000", "WILL5000IND", "Wilshire 5000 Total Market Index", "Index"),
]

fred_failed = []
for series_id, fred_id, description, units in FRED_SERIES:
    try:
        adapter = FREDAdapter(fred_id)
        meta = SeriesMetadata(
            series_id=series_id,
            description=description,
            source=f"FRED ({fred_id})",
            units=units,
            frequency="MS",
        )
        svc.register(series_id, adapter, meta)
    except Exception as e:
        fred_failed.append((series_id, str(e)))
        print(f"  [WARN] {series_id}: {e}")

if fred_failed:
    print(f"\n{len(fred_failed)} FRED series failed. Check FRED_API_KEY and series IDs.")
else:
    print(f"Registered {len(FRED_SERIES)} FRED covariate series.")

## 3. Inspect Coverage

In [ ]:
as_of_now = datetime.now(tz=timezone.utc).replace(tzinfo=None)

summary_all = svc.summary()
summary_all["start"] = summary_all["start"].dt.strftime("%Y-%m")
summary_all["end"] = summary_all["end"].dt.strftime("%Y-%m")
summary_all

## 4. Visualise Food CPI History

Plot the 9 food CPI categories together to understand relative levels, trends, and seasonal patterns.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

FOOD_SERIES_IDS = [sid for sid, _, _ in FOOD_CPI_SERIES]
FOOD_LABELS = {
    "cpi_food_canada": "Overall Food",
    "cpi_bakery_cereal_canada": "Bakery & Cereal",
    "cpi_dairy_eggs_canada": "Dairy & Eggs",
    "cpi_fish_seafood_canada": "Fish & Seafood",
    "cpi_restaurants_canada": "Restaurants",
    "cpi_fruit_preparations_nuts_canada": "Fruit & Nuts",
    "cpi_meat_canada": "Meat",
    "cpi_other_food_nonalcoholic_canada": "Other Food",
    "cpi_vegetables_preparations_canada": "Vegetables",
}

fig, axes = plt.subplots(3, 3, figsize=(16, 12), sharex=True)
axes_flat = axes.flatten()

for ax, series_id in zip(axes_flat, FOOD_SERIES_IDS):
    try:
        df = svc.get_series(series_id, as_of=as_of_now)
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        ax.plot(df["timestamp"], df["value"], linewidth=1.2, color="#2563eb")
        ax.set_title(FOOD_LABELS[series_id], fontsize=10, fontweight="bold")
        ax.set_ylabel("Index (2002=100)", fontsize=8)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.xaxis.set_major_locator(mdates.YearLocator(5))
        ax.tick_params(axis="x", labelrotation=45, labelsize=8)
    except Exception as e:
        ax.text(0.5, 0.5, f"Not available\n{e}", ha="center", va="center", transform=ax.transAxes, fontsize=9)
        ax.set_title(FOOD_LABELS[series_id], fontsize=10, fontweight="bold")

fig.suptitle("Canada Food CPI Sub-categories (2002=100)", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# Year-over-year % change for all food categories — shows the dynamics we're forecasting.
fig, axes = plt.subplots(3, 3, figsize=(16, 12), sharex=True, sharey=True)
axes_flat = axes.flatten()

for ax, series_id in zip(axes_flat, FOOD_SERIES_IDS):
    try:
        df = svc.get_series(series_id, as_of=as_of_now)
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df.set_index("timestamp").sort_index()
        yoy = df["value"].pct_change(12) * 100
        ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
        ax.fill_between(yoy.index, yoy.values, 0, where=(yoy.values > 0), alpha=0.3, color="#dc2626")
        ax.fill_between(yoy.index, yoy.values, 0, where=(yoy.values <= 0), alpha=0.3, color="#16a34a")
        ax.plot(yoy.index, yoy.values, linewidth=1.0, color="#374151")
        ax.set_title(FOOD_LABELS[series_id], fontsize=10, fontweight="bold")
        ax.set_ylabel("YoY %", fontsize=8)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.xaxis.set_major_locator(mdates.YearLocator(5))
        ax.tick_params(axis="x", labelrotation=45, labelsize=8)
    except Exception as e:
        ax.text(0.5, 0.5, f"Not available\n{e}", ha="center", va="center", transform=ax.transAxes, fontsize=9)
        ax.set_title(FOOD_LABELS[series_id], fontsize=10, fontweight="bold")

fig.suptitle("Canada Food CPI Year-over-Year % Change", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## 5. Visualise FRED Covariates

Inspect the exogenous indicators and their relationship with Overall Food CPI.

In [ ]:
FRED_SERIES_IDS = [sid for sid, _, _, _ in FRED_SERIES]
FRED_LABELS = {
    "fred_us_cpi_food_at_home": "US CPI: Food at Home",
    "fred_us_cpi_meats_poultry_fish_eggs": "US CPI: Meats/Poultry/Fish/Eggs",
    "fred_us_cpi_fruits_vegetables": "US CPI: Fruits & Vegetables",
    "fred_canada_10yr_bond_yield": "Canada 10yr Bond Yield (%)",
    "fred_canada_us_exchange_rate": "CAD/USD Exchange Rate",
    "fred_sp100_volatility_vxo": "VXO (S&P 100 Volatility)",
    "fred_wilshire_5000": "Wilshire 5000 Index",
}

n_fred = len(FRED_SERIES_IDS)
fig, axes = plt.subplots(n_fred, 1, figsize=(14, 3 * n_fred))
if n_fred == 1:
    axes = [axes]

for ax, series_id in zip(axes, FRED_SERIES_IDS):
    try:
        df = svc.get_series(series_id, as_of=as_of_now)
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        ax.plot(df["timestamp"], df["value"], linewidth=1.2, color="#7c3aed")
        ax.set_title(FRED_LABELS[series_id], fontsize=10, fontweight="bold")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.xaxis.set_major_locator(mdates.YearLocator(5))
        ax.tick_params(axis="x", labelrotation=45, labelsize=8)
    except Exception as e:
        ax.text(0.5, 0.5, f"Not available\n{e}", ha="center", va="center", transform=ax.transAxes, fontsize=9)
        ax.set_title(FRED_LABELS[series_id], fontsize=10, fontweight="bold")

fig.suptitle("FRED Exogenous Covariates", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# Cross-correlation: FRED covariates vs. Overall Food CPI YoY
# Helps identify which covariates are potentially useful and at what lag.
import numpy as np

food_df = svc.get_series("cpi_food_canada", as_of=as_of_now)
food_df["timestamp"] = pd.to_datetime(food_df["timestamp"])
food_yoy = food_df.set_index("timestamp")["value"].pct_change(12).dropna()

LAGS = [0, 3, 6, 12]
corr_rows = []

for series_id in FRED_SERIES_IDS:
    try:
        df = svc.get_series(series_id, as_of=as_of_now)
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        cov_yoy = df.set_index("timestamp")["value"].pct_change(12).dropna()
        row = {"series": FRED_LABELS[series_id]}
        for lag in LAGS:
            aligned = pd.concat([food_yoy, cov_yoy.shift(lag)], axis=1).dropna()
            if len(aligned) > 10:
                row[f"lag_{lag}m"] = round(aligned.corr().iloc[0, 1], 3)
            else:
                row[f"lag_{lag}m"] = None
        corr_rows.append(row)
    except Exception:
        pass

if corr_rows:
    corr_df = pd.DataFrame(corr_rows).set_index("series")
    corr_df.columns = [f"Lag {lag}m" for lag in LAGS]
    corr_df.style.background_gradient(cmap="RdBu", axis=None, vmin=-1, vmax=1)